<center><img src="https://github.com/hse-ds/iad-applied-ds/blob/master/2021/hw/hw1/img/logo_hse.png?raw=1" width="1000"></center>

<h1><center>Прикладные задачи анализа данных</center></h1>
<h2><center>Домашнее задание 4: рекомендательные системы</center></h2>

# Введение

В этом задании Вы продолжите работать с данными из семинара [Articles Sharing and Reading from CI&T Deskdrop](https://www.kaggle.com/gspmoreira/articles-sharing-reading-from-cit-deskdrop).

# Загрузка и предобработка данных

In [1]:
import pandas as pd
import numpy as np
import math


Загрузим данные и проведем предобраотку данных как на семинаре.

In [2]:
# !kaggle datasets download -d gspmoreira/articles-sharing-reading-from-cit-deskdrop
# !unzip articles-sharing-reading-from-cit-deskdrop.zip -d articles


In [3]:
articles_df = pd.read_csv("shared_articles.csv")
articles_df = articles_df[articles_df["eventType"] == "CONTENT SHARED"]
articles_df.head(2)


,timestamp,eventType,contentId,authorPersonId,authorSessionId,authorUserAgent,authorRegion,authorCountry,contentType,url,title,text,lang
1,1459193988,CONTENT SHARED,-4110354420726924665,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,http://www.nytimes.com/2016/03/28/business/dea...,"Ethereum, a Virtual Currency, Enables Transact...",All of this work is still very early. The firs...,en
2,1459194146,CONTENT SHARED,-7292285110016212249,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,http://cointelegraph.com/news/bitcoin-future-w...,Bitcoin Future: When GBPcoin of Branson Wins O...,The alarm clock wakes me at 8:00 with stream o...,en


In [4]:
interactions_df = pd.read_csv("users_interactions.csv")
interactions_df.head(2)


,timestamp,eventType,contentId,personId,sessionId,userAgent,userRegion,userCountry
0,1465413032,VIEW,-3499919498720038879,-8845298781299428018,1264196770339959068,NaN,NaN,NaN
1,1465412560,VIEW,8890720798209849691,-1032019229384696495,3621737643587579081,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,NY,US


In [5]:
interactions_df.personId = interactions_df.personId.astype(str)
interactions_df.contentId = interactions_df.contentId.astype(str)
articles_df.contentId = articles_df.contentId.astype(str)


In [6]:
# зададим словарь определяющий силу взаимодействия
event_type_strength = {
    "VIEW": 1.0,
    "LIKE": 2.0,
    "BOOKMARK": 2.5,
    "FOLLOW": 3.0,
    "COMMENT CREATED": 4.0,
}

interactions_df["eventStrength"] = interactions_df.eventType.apply(
    lambda x: event_type_strength[x]
)


Оставляем только тех пользователей, которые произамодействовали более чем с пятью статьями.

In [7]:
users_interactions_count_df = (
    interactions_df.groupby(["personId", "contentId"])
    .first()
    .reset_index()
    .groupby("personId")
    .size()
)
print("# users:", len(users_interactions_count_df))

users_with_enough_interactions_df = users_interactions_count_df[
    users_interactions_count_df >= 5
].reset_index()[["personId"]]
print("# users with at least 5 interactions:", len(users_with_enough_interactions_df))


# users: 1895
# users with at least 5 interactions: 1140


Оставляем только те взаимодействия, которые относятся к отфильтрованным пользователям.

In [8]:
interactions_from_selected_users_df = interactions_df.loc[
    np.in1d(interactions_df.personId, users_with_enough_interactions_df)
]


In [9]:
print(f"# interactions before: {interactions_df.shape}")
print(f"# interactions after: {interactions_from_selected_users_df.shape}")


# interactions before: (72312, 9)
# interactions after: (69868, 9)


Объединяем все взаимодействия пользователя по каждой статье и сглажиываем полученный результат, взяв от него логарифм.

In [10]:
def smooth_user_preference(x):
    return math.log(1 + x, 2)


interactions_full_df = (
    interactions_from_selected_users_df.groupby(["personId", "contentId"])
    .eventStrength.sum()
    .apply(smooth_user_preference)
    .reset_index()
    .set_index(["personId", "contentId"])
)
interactions_full_df["last_timestamp"] = interactions_from_selected_users_df.groupby(
    ["personId", "contentId"]
)["timestamp"].last()

interactions_full_df = interactions_full_df.reset_index()
interactions_full_df.head(5)


,personId,contentId,eventStrength,last_timestamp
0,-1007001694607905623,-5065077552540450930,1.000000,1470395911
1,-1007001694607905623,-6623581327558800021,1.000000,1487240080
2,-1007001694607905623,-793729620925729327,1.000000,1472834892
3,-1007001694607905623,1469580151036142903,1.000000,1487240062
4,-1007001694607905623,7270966256391553686,1.584963,1485994324


Разобьём выборку на обучение и контроль по времени.

In [11]:
from sklearn.model_selection import train_test_split

split_ts = 1475519530
interactions_train_df = interactions_full_df.loc[
    interactions_full_df.last_timestamp < split_ts
].copy()
interactions_test_df = interactions_full_df.loc[
    interactions_full_df.last_timestamp >= split_ts
].copy()

print(f"# interactions on Train set: {len(interactions_train_df)}")
print(f"# interactions on Test set: {len(interactions_test_df)}")

interactions_train_df


# interactions on Train set: 29329
# interactions on Test set: 9777


,personId,contentId,eventStrength,last_timestamp
0,-1007001694607905623,-5065077552540450930,1.0,1470395911
2,-1007001694607905623,-793729620925729327,1.0,1472834892
6,-1032019229384696495,-1006791494035379303,1.0,1469129122
7,-1032019229384696495,-1039912738963181810,1.0,1459376415
8,-1032019229384696495,-1081723567492738167,2.0,1464054093
...,...,...,...,...
39099,997469202936578234,9112765177685685246,2.0,1472479493
39100,998688566268269815,-1255189867397298842,1.0,1474567164
39101,998688566268269815,-401664538366009049,1.0,1474567449
39103,998688566268269815,6881796783400625893,1.0,1474567675


Для удобства подсчёта качества запишем данные в формате, где строка соответствует пользователю, а столбцы будут истинными метками и предсказаниями в виде списков.

In [12]:
interactions = (
    interactions_train_df.groupby("personId")["contentId"]
    .agg(lambda x: list(x))
    .reset_index()
    .rename(columns={"contentId": "true_train"})
    .set_index("personId")
)

interactions["true_test"] = interactions_test_df.groupby("personId")["contentId"].agg(
    lambda x: list(x)
)

# заполнение пропусков пустыми списками
interactions.loc[pd.isnull(interactions.true_test), "true_test"] = [
    ""
    for x in range(
        len(interactions.loc[pd.isnull(interactions.true_test), "true_test"])
    )
]

interactions.head(1)


,true_train,true_test
personId,,
-1007001694607905623,"[-5065077552540450930, -793729620925729327]","[-6623581327558800021, 1469580151036142903, 72..."


# Библиотека LightFM

Для рекомендации Вы будете пользоваться библиотекой [LightFM](https://making.lyst.com/lightfm/docs/home.html), в которой реализованы популярные алгоритмы. Для оценивания качества рекомендации, как и на семинаре, будем пользоваться метрикой *precision@10*.

In [13]:
# !pip install lightfm


In [14]:
from lightfm import LightFM
from lightfm.evaluation import precision_at_k


## Задание 1 (2 балла)

Модели в LightFM работают с разреженными матрицами. Создайте разреженные матрицы `data_train` и `data_test` (размером количество пользователей на количество статей), такие что на пересечении строки пользователя и столбца статьи стоит сила их взаимодействия, если взаимодействие было, и стоит ноль, если взаимодействия не было.

In [15]:
ratings_full = pd.pivot_table(
    interactions_full_df, values="eventStrength", index="personId", columns="contentId"
).fillna(0)
ratings_full


contentId,-1006791494035379303,-1021685224930603833,-1022885988494278200,-1024046541613287684,-1033806831489252007,-1038011342017850,-1039912738963181810,-1046621686880462790,-1051830303851697653,-1055630159212837930,...,9222265156747237864,943818026930898372,957332268361319692,962287586799267519,966067567430037498,967143806332397325,972258375127367383,980458131533897249,98528655405030624,991271693336573226
personId,,,,,,,,,,,,,,,,,,,,,
-1007001694607905623,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
-1032019229384696495,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,2.321928,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
-108842214936804958,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
-1119397949556155765,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
-1130272294246983140,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
953707509720613429,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
983095443598229476,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
989049974880576288,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
ratings_train = pd.pivot_table(
    interactions_train_df, values="eventStrength", index="personId", columns="contentId"
).fillna(0)
ratings_train


contentId,-1006791494035379303,-1021685224930603833,-1022885988494278200,-1024046541613287684,-1033806831489252007,-1038011342017850,-1039912738963181810,-1046621686880462790,-1051830303851697653,-1055630159212837930,...,9217155070834564627,921770761777842242,9220445660318725468,9222265156747237864,943818026930898372,957332268361319692,966067567430037498,972258375127367383,980458131533897249,98528655405030624
personId,,,,,,,,,,,,,,,,,,,,,
-1007001694607905623,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
-1032019229384696495,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,3.0,0.0,0.0,0.0,2.321928,0.0,0.0,0.0,0.0,0.0
-108842214936804958,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,2.0,0.0,0.0,0.0
-1130272294246983140,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.000000,0.0,0.0,0.0,0.0,0.0
-1160159014793528221,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
953707509720613429,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
983095443598229476,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
989049974880576288,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0


In [17]:
ratings_test = pd.pivot_table(
    interactions_test_df, values="eventStrength", index="personId", columns="contentId"
).fillna(0)
ratings_test


contentId,-1021685224930603833,-1022885988494278200,-1072987232233605661,-1101361754763388054,-1104501717379772664,-1119244241345534741,-1123543351704082417,-1124738136890721085,-1129449063360470561,-1138633255366005559,...,9208127165664287660,9209629151177723638,9209886322932807692,9213260650272029784,921770761777842242,9220445660318725468,962287586799267519,966067567430037498,967143806332397325,991271693336573226
personId,,,,,,,,,,,,,,,,,,,,,
-1007001694607905623,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
-1032019229384696495,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
-108842214936804958,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
-1119397949556155765,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
-1130272294246983140,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
953707509720613429,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
983095443598229476,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
989049974880576288,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
from scipy.sparse import csr_matrix

data_train = csr_matrix(ratings_full[ratings_full.isin(ratings_train)].fillna(0).values)
data_test = csr_matrix(ratings_full[ratings_full.isin(ratings_test)].fillna(0).values)
print(data_train.shape, data_test.shape)


(1140, 2984) (1140, 2984)


## Задание 2 (1 балл)

Обучите модель LightFM с `loss="warp"` и посчитайте *precision@10* на тесте.

In [19]:
model = LightFM(loss="warp")
model.fit(data_train, epochs=20)
train_precision = precision_at_k(model, data_train, k=10).mean()
test_precision = precision_at_k(
    model, data_test, k=10, train_interactions=data_train
).mean()
print(f"train: {train_precision}\ntest: {test_precision}")


train: 0.23012590408325195
test: 0.007331975270062685


## Задание 3 (3 балла)

При вызове метода `fit` LightFM позволяет передавать в `item_features` признаковое описание объектов. Воспользуемся этим. Будем получать признаковое описание из текста статьи в виде [TF-IDF](https://ru.wikipedia.org/wiki/TF-IDF) (можно воспользоваться `TfidfVectorizer` из scikit-learn). Создайте матрицу `feat` размером количесвто статей на размер признакового описание и обучите LightFM с `loss="warp"` и посчитайте precision@10 на тесте.

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
feat = vectorizer.fit_transform(articles_df["text"])

model = LightFM(loss="warp")
model.fit(data_train, item_features=feat, epochs=20)
train_precision = precision_at_k(model, data_train, item_features=feat, k=10).mean()
test_precision = precision_at_k(
    model, data_test, k=10, train_interactions=data_train, item_features=feat
).mean()
print(f"train: {train_precision}\ntest: {test_precision}")


train: 0.23390288650989532
test: 0.00529531529173255


## Задание 4 (2 балла)

В задании 3 мы использовали сырой текст статей. В этом задании необходимо сначала сделать предобработку текста (привести к нижнему регистру, убрать стоп слова, привести слова к номральной форме и т.д.), после чего обучите модель и оценить качество на тестовых данных.

In [21]:
# Соединю заголовок и текст
articles_df["title+text"] = articles_df["title"] + " " + articles_df["text"]


from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
from string import punctuation
from string import digits

sw = set(stopwords.words("english") + stopwords.words("portuguese"))
stemmer = SnowballStemmer("english")
noise = set(punctuation + digits)


def MyTokenizer(sentence):
    tokens = [
        stemmer.stem(x)
        for x in word_tokenize(sentence.lower())
        if (len(set(x) & noise) == 0 and len(set([x]) & sw) == 0 and len(x) > 2)
    ]

    # Так же буду ещё отбрасывать короткие слова имеющие длину 2 символа и меньше
    return tokens


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(tokenizer=MyTokenizer, ngram_range=(1, 2))
feat = vectorizer.fit_transform(articles_df["title+text"])

model = LightFM(loss="warp")
model.fit(data_train, item_features=feat, epochs=20)
train_precision = precision_at_k(model, data_train, item_features=feat, k=10).mean()
test_precision = precision_at_k(
    model, data_test, k=10, train_interactions=data_train, item_features=feat
).mean()
print(f"train: {train_precision}\ntest: {test_precision}")


train: 0.2506295144557953
test: 0.006720977835357189


Улучшилось ли качество предсказания? Нет))

## Задание 5 (2 балла)

Подберите гиперпараметры модели LightFM (`n_components` и др.) для улучшения качества модели.

In [23]:
from sklearn.model_selection import ParameterGrid

parameters = ParameterGrid(
    {
        "no_components": (10, 20, 30, 40, 50, 60),
        "learning_schedule": ("adagrad", "adadelta"),
        "learning_rate": (0.05, 0.1, 0.01, 0.001),
    }
)
train_precisions = []
test_precisions = []

for params in parameters:
    model = LightFM(**params, loss="warp")
    model.fit(data_train, item_features=feat, epochs=20)
    train_precision = precision_at_k(model, data_train, item_features=feat, k=10).mean()
    test_precision = precision_at_k(
        model, data_test, k=10, train_interactions=data_train, item_features=feat
    ).mean()
    train_precisions.append(train_precision)
    test_precisions.append(test_precision)
    print(f"params: {params}\ntrain: {train_precision} test: {test_precision}")


params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 10}
train: 0.23803958296775818 test: 0.006211812607944012
params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 20}
train: 0.3386690616607666 test: 0.007535641547292471
params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 30}
train: 0.4285971224308014 test: 0.006720977835357189
params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 40}
train: 0.5081834197044373 test: 0.006720977835357189
params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 50}
train: 0.5791366696357727 test: 0.007637474685907364
params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 60}
train: 0.6202338337898254 test: 0.006109979469329119
params: {'learning_rate': 0.05, 'learning_schedule': 'adadelta', 'no_components': 10}
train: 0.23102517426013947 test: 0.007433808874338865
params: {'learning_rate': 0.05,

In [24]:
# Лучшие параметры
max_index = test_precisions.index(max(test_precisions))
print(f"train: {train_precisions[max_index]}\ntest: {test_precisions[max_index]}")
print(parameters[max_index])


train: 0.22922663390636444
test: 0.008757637813687325
{'no_components': 40, 'learning_schedule': 'adagrad', 'learning_rate': 0.01}


## Бонусное задание (3 балла)

Выше мы использовали достаточно простое представление текста статьи в виде TF-IDF. В этом задании Вам нужно представить текст статьи (можно вместе с заголовком) в виде эмбеддинга полученного с помощью рекуррентной сети или трансформера (можно использовать любую предобученную модель, которая Вам нравится). Обучите модель с ипользованием этих эмеддингов и сравните результаты с предыдущими.

In [25]:
from sentence_transformers import SentenceTransformer

# Текст статей на разном языке, используем мульти-лингвистическую модель
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = csr_matrix(model.encode(articles_df["title+text"].tolist()))

parameters = ParameterGrid(
    {
        "no_components": (10, 20, 30, 40, 50, 60),
        "learning_schedule": ("adagrad", "adadelta"),
        "learning_rate": (0.05, 0.1, 0.01, 0.001),
    }
)
train_precisions = []
test_precisions = []

for params in parameters:
    model = LightFM(**params, loss="warp")
    model.fit(data_train, item_features=embeddings, epochs=20)
    train_precision = precision_at_k(
        model, data_train, item_features=embeddings, k=10
    ).mean()
    test_precision = precision_at_k(
        model, data_test, k=10, train_interactions=data_train, item_features=embeddings
    ).mean()
    train_precisions.append(train_precision)
    test_precisions.append(test_precision)
    print(f"params: {params}\ntrain: {train_precision} test: {test_precision}")


params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 10}
train: 0.06636691093444824 test: 0.0028513239230960608
params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 20}
train: 0.09955035895109177 test: 0.0025458247400820255
params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 30}
train: 0.11906474083662033 test: 0.0031568228732794523
params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 40}
train: 0.1379496306180954 test: 0.0026476578786969185
params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 50}
train: 0.16429857909679413 test: 0.0032586560118943453
params: {'learning_rate': 0.05, 'learning_schedule': 'adagrad', 'no_components': 60}
train: 0.18669067323207855 test: 0.0031568228732794523
params: {'learning_rate': 0.05, 'learning_schedule': 'adadelta', 'no_components': 10}
train: 0.07517985999584198 test: 0.004175153095275164
params: {'learning_ra

In [26]:
# Лучшие параметры
max_index = test_precisions.index(max(test_precisions))
print(f"train: {train_precisions[max_index]}\ntest: {test_precisions[max_index]}")
print(parameters[max_index])


train: 0.08579137176275253
test: 0.00539714889600873
{'no_components': 10, 'learning_schedule': 'adadelta', 'learning_rate': 0.01}
